In [1]:
import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100

# Здесь инициализируется подключение к ClickHouse 
database = 'dev'
schema = 'artemw9'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))



In [2]:
client.command(f'''
    CREATE DATABASE IF NOT EXISTS {database} ON CLUSTER '{{cluster}}'; 
''')


['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

In [3]:
client.command(f'DROP TABLE IF EXISTS {schema}.cars_distr ON CLUSTER company_cluster;')
client.command(f'DROP TABLE IF EXISTS {schema}.cars ON CLUSTER company_cluster;')

['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

In [5]:
client.command(f'''
    CREATE TABLE {schema}.cars ON CLUSTER 'company_cluster'
    (
        car_id      UInt32,
        car_vin     String,
        cost        UInt32,
        date_of_pay DateTime
    )
    ENGINE = ReplicatedMergeTree('/clickhouse/tables/{{cluster}}/{{shard}}/cars', '{{replica}}')
    PARTITION BY toDate(date_of_pay)
    ORDER BY (car_id);
''')



['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

In [6]:
client.command(f'''
    CREATE TABLE {schema}.cars_distr ON CLUSTER 'company_cluster' 
    AS {schema}.cars
    ENGINE = Distributed('company_cluster', {schema}, cars, car_id);
''')

['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

In [ ]:
client.command(f'''
    INSERT INTO {schema}.cars_distr(car_id, car_vin, cost, date_of_pay) VALUES
        (1, '1HGCM82633A123456', 2500000, '2020-01-01 10:00:00'),
        (2, 'JTDKB20U987654321', 1800000, '2020-01-01 10:05:00'),
        (3, 'WDDGF8BB7AF123456', 3500000, '2020-01-01 11:00:00'),
        (4, 'ZFF67ALA5F0123456', 1200000, '2020-01-01 12:10:00'),
        (5, 'WAUZZZ8V7BA123456', 2800000, '2020-01-02 08:10:00'),
        (6, 'VF1RFD00612345678', 1700000, '2020-01-03 13:00:00')
''')


In [9]:
client.query_df(f'''
    select * from {schema}.cars_distr 
''')

,car_id,car_vin,cost,date_of_pay
0,6,VF1RFD00612345678,1700000,2020-01-03 13:00:00+03:00
1,2,JTDKB20U987654321,1800000,2020-01-01 10:05:00+03:00
2,4,ZFF67ALA5F0123456,1200000,2020-01-01 12:10:00+03:00
3,5,WAUZZZ8V7BA123456,2800000,2020-01-02 08:10:00+03:00
4,1,1HGCM82633A123456,2500000,2020-01-01 10:00:00+03:00
5,3,WDDGF8BB7AF123456,3500000,2020-01-01 11:00:00+03:00


In [10]:
# Проверить, какие макросы определены
print("Макросы на сервере:")
macros = client.query_df("SELECT * FROM system.macros")
print(macros)

Макросы на сервере:
     macro     substitution
0  cluster  company_cluster
1  replica     clickhouse01
2    shard               01


In [11]:
# Здесь создаентся подключение к PostgreSQL к схеме artemw9
import psycopg2 as ps
import pandas as pd
import os

schema = 'artemw9' 

conn = ps.connect(host="postgres_source", 
                  port = 5432, 
                  database="dev", 
                  user=os.getenv("POSTGRES_USER"), 
                  password=os.getenv("POSTGRES_PASSWORD"), 
                  )

cursor = conn.cursor()


In [12]:

cursor.execute(f'''
    CREATE SCHEMA IF NOT EXISTS {schema};
    ''')

cursor.execute(f'''
    DROP TABLE IF EXISTS {schema}.cars_table CASCADE;
''')

cursor.execute(f'''
    CREATE TABLE {schema}.cars_table (
        car_id  SERIAL PRIMARY KEY,
        car_vin VARCHAR(50),
        cost Integer,
        date_of_pay TIMESTAMP
    )
''')

cursor.execute(f'''
    INSERT INTO {schema}.cars_table (car_id, car_vin,cost,date_of_pay) VALUES
        (7, 'SALYA2BNXKA123456', 3200000, '2020-01-04 09:30:00'),
        (8, 'WBA8E1G59J1234567', 2900000, '2020-01-05 14:20:00'),
        (9, 'KMHCT41CPPU123456', 2100000, '2020-01-06 11:15:00'),
        (10, '3VWLL7AJ2BM123456', 1950000, '2020-01-07 16:45:00'),
        (11, '5YJSA1E26KF123456', 4500000, '2020-01-08 10:30:00'),
        (12, '1G1ZE5ST0JF123456', 2300000, '2020-01-09 09:15:00'),
        (13, '2C3CDXJG2JH123456', 3100000, '2020-01-10 14:50:00'),
        (14, 'JM1BL1VF3B1234567', 1600000, '2020-01-11 11:30:00'),
        (15, 'WA1VAAF74KD123456', 3800000, '2020-01-12 13:20:00'),
        (16, '1N4AL3AP9JC123456', 2700000, '2020-01-13 15:40:00'),
        (17, '3FA6P0H71HR123456', 2200000, '2020-01-14 08:50:00'),
        (18, '5NPE34AFXFH123456', 1900000, '2020-01-15 12:10:00'),
        (19, 'JN8AZ2NC8B9123456', 3300000, '2020-01-16 10:00:00'),
        (20, '1C4RJFAG0EC123456', 2600000, '2020-01-17 09:45:00');
''')
conn.commit()



In [13]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.postgresql_table;
''')

client.command(f'''
    CREATE TABLE {database}.postgresql_table
    (
        car_id  UInt32,
        car_vin String,
        cost UInt32,
        date_of_pay Datetime
    )
    ENGINE = PostgreSQL(
                        'postgres_source:5432', -- сервер, порт
                        'dev',              -- БД 
                        'cars_table',      -- таблица
                        '{os.getenv("POSTGRES_USER")}',         -- логин
                        '{os.getenv("POSTGRES_PASSWORD")}',         -- пароль
                        'artemw9'         -- имя схемы
                        );
''')

In [14]:
client.query_df(f'''
     select * from {database}.postgresql_table
''')

,car_id,car_vin,cost,date_of_pay
0,7,SALYA2BNXKA123456,3200000,2020-01-04 09:30:00+03:00
1,8,WBA8E1G59J1234567,2900000,2020-01-05 14:20:00+03:00
2,9,KMHCT41CPPU123456,2100000,2020-01-06 11:15:00+03:00
3,10,3VWLL7AJ2BM123456,1950000,2020-01-07 16:45:00+03:00
4,11,5YJSA1E26KF123456,4500000,2020-01-08 10:30:00+03:00
5,12,1G1ZE5ST0JF123456,2300000,2020-01-09 09:15:00+03:00
6,13,2C3CDXJG2JH123456,3100000,2020-01-10 14:50:00+03:00
7,14,JM1BL1VF3B1234567,1600000,2020-01-11 11:30:00+03:00
8,15,WA1VAAF74KD123456,3800000,2020-01-12 13:20:00+03:00
9,16,1N4AL3AP9JC123456,2700000,2020-01-13 15:40:00+03:00


In [15]:
client.command(f'''
    INSERT INTO {schema}.cars_distr 
    SELECT * FROM {database}.postgresql_table
''')

In [16]:
client.query_df(f'''
    select * from {schema}.cars_distr 
    order by car_id
''')

,car_id,car_vin,cost,date_of_pay
0,1,1HGCM82633A123456,2500000,2020-01-01 10:00:00+03:00
1,2,JTDKB20U987654321,1800000,2020-01-01 10:05:00+03:00
2,3,WDDGF8BB7AF123456,3500000,2020-01-01 11:00:00+03:00
3,4,ZFF67ALA5F0123456,1200000,2020-01-01 12:10:00+03:00
4,5,WAUZZZ8V7BA123456,2800000,2020-01-02 08:10:00+03:00
5,6,VF1RFD00612345678,1700000,2020-01-03 13:00:00+03:00
6,7,SALYA2BNXKA123456,3200000,2020-01-04 09:30:00+03:00
7,8,WBA8E1G59J1234567,2900000,2020-01-05 14:20:00+03:00
8,9,KMHCT41CPPU123456,2100000,2020-01-06 11:15:00+03:00
9,10,3VWLL7AJ2BM123456,1950000,2020-01-07 16:45:00+03:00


In [ ]:
# Т к шард всего один, то запрос к cars_distr перенаправляет к сars